# COJO support for additional credible sets

COJO estimates conditional associations using GWAS summary statistics and reference LD. 

We use [cojopy](https://github.com/Jianhua-Wang/cojopy) to ask whether credible sets with weak marginal associations have joint support. This implementation accepts the supplied LD matrix.

We approximate effect estimates and standard errors from the $z$-scores using an allele frequency of 0.5, because allele frequencies are missing in some summary files. 

## Weak credible sets

In [3]:
library(tidyverse)
library(pecotmr)
devtools::load_all("/hpc/users/sunh17/haochen/rss_qc/pecotmr_latest/pecotmr", compile = FALSE)
read_gwas_tables <- function(folder) {
  
  files <- list.files(
    path = folder,
    pattern = "\\.gwas\\.top_loci\\.bed\\.gz$",
    full.names = TRUE
  )
  
  map_dfr(files, function(file) {
    
    study_name <- basename(file) |>
      str_remove("\\.gwas\\.top_loci\\.bed\\.gz$")
    
    read_delim(
      file,
      delim = "\t",
      show_col_types = FALSE
    ) |>
      mutate(study = study_name)
  })
}

eb_mix = read_gwas_tables("/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10/AD_GWAS_rss_ebmix_r4_L10/export/")
eb_mix = eb_mix %>% mutate(method = "eb_mix") %>% filter(cs_coverage_0.95 != "susie_rss_0")



Attaching package: ‘pecotmr’


The following objects are masked _by_ ‘.GlobalEnv’:

    dentist, raiss, sdpr, slalom


ℹ Loading pecotmr
Creating a generic function for ‘endoapply’ from package ‘S4Vectors’ in package ‘pecotmr’

Warning message:
“Failed to load at least one DLL.
Caused by error in `getDLLRegisteredRoutines.DLLInfo()`:
! must specify DLL via a “DLLInfo” object. See getLoadedDLLs()”
Warning message:
“── Conflicts ────────────────────────────────────────────── pecotmr conflicts
──
✖ `dentist` masks `pecotmr::dentist()`.
✖ `raiss` masks `pecotmr::raiss()`.
✖ `sdpr` masks `pecotmr::sdpr()`.
✖ `slalom` masks `pecotmr::slalom()`.
ℹ Did you accidentally source a file rather than using `load_all()`?
  Run `rm(list = c("dentist", "raiss", "sdpr", "slalom"))` to remove the
  conflicts.”


In [4]:
library(tidyverse)

## one row per credible set, with its strongest |z|
cs_tab <- eb_mix %>%
  filter(!is.na(cs_coverage_0.95), !cs_coverage_0.95 %in% c("0", "susie_rss_0")) %>%
  group_by(study, block_id, cs = cs_coverage_0.95) %>%
  summarise(max_abs_z = max(abs(z), na.rm = TRUE), .groups = "drop") %>%
  mutate(p_of_maxz = 2 * pnorm(-max_abs_z))

## counts at each threshold
thr <- c(1e-3, 1e-4, 1e-5)
summ <- tibble(p_thresh = thr, z_thresh = qnorm(1 - thr/2)) %>%
  rowwise() %>%
  mutate(
    n_weak_cs     = sum(cs_tab$max_abs_z < z_thresh),
    n_weak_region = n_distinct(paste(cs_tab$study, cs_tab$block_id)[cs_tab$max_abs_z < z_thresh])
  ) %>% ungroup()
summ

p_thresh,z_thresh,n_weak_cs,n_weak_region
<dbl>,<dbl>,<int>,<int>
1e-03,3.290527,4,4
1e-04,3.890592,5,5
1e-05,4.417173,12,11


In [5]:
cs_tab <- cs_tab %>%
  group_by(study, block_id) %>%
  mutate(n_cs_total = n_distinct(cs)) %>%     # total CS in that study × region
  ungroup()

weak_list <- function(p_thresh) {
  z <- qnorm(1 - p_thresh/2)
  cs_tab %>%
    filter(max_abs_z < z) %>%
    arrange(study, block_id, cs) %>%
    transmute(study, block_id, cs,
              max_abs_z  = round(max_abs_z, 3),
              p_of_maxz  = signif(p_of_maxz, 3),
              n_cs_total)                      # how many CS total for this region × study
}

weak_list(1e-3)
weak_list(1e-4)
weak_list(1e-5)

study,block_id,cs,max_abs_z,p_of_maxz,n_cs_total
<chr>,<chr>,<chr>,<dbl>,<dbl>,<int>
AD_Bellenguez_2022,chr2_125689597_127728648,susie_rss_2,2.926,0.00344,2
AD_Bellenguez_EADB_2022,chr19_90949_1507536,susie_rss_2,3.033,0.00242,2
AD_Bellenguez_main_2026,chr2_125689597_127728648,susie_rss_2,2.728,0.00637,2
AD_Bellenguez_noproxy_2026,chr2_125689597_127728648,susie_rss_2,2.867,0.00415,2


study,block_id,cs,max_abs_z,p_of_maxz,n_cs_total
<chr>,<chr>,<chr>,<dbl>,<dbl>,<int>
AD_Bellenguez_2022,chr2_125689597_127728648,susie_rss_2,2.926,0.003440,2
AD_Bellenguez_EADB_2022,chr19_90949_1507536,susie_rss_2,3.033,0.002420,2
AD_Bellenguez_main_2026,chr2_125689597_127728648,susie_rss_2,2.728,0.006370,2
AD_Bellenguez_noproxy_2026,chr2_125689597_127728648,susie_rss_2,2.867,0.004150,2
AD_Wightman_ExcludingUKBand23andME_2021,chr8_143034200_145075767,susie_rss_2,3.878,0.000105,2


study,block_id,cs,max_abs_z,p_of_maxz,n_cs_total
<chr>,<chr>,<chr>,<dbl>,<dbl>,<int>
AD_Bellenguez_2022,chr2_125689597_127728648,susie_rss_2,2.926,3.44e-03,2
AD_Bellenguez_EADB_2022,chr19_90949_1507536,susie_rss_2,3.033,2.42e-03,2
AD_Bellenguez_main_2026,chr2_125689597_127728648,susie_rss_2,2.728,6.37e-03,2
AD_Bellenguez_nobiobank_2026,chr11_119829118_122612025,susie_rss_2,4.247,2.17e-05,2
AD_Bellenguez_nobiobank_2026,chr19_53788025_55215821,susie_rss_3,4.200,2.67e-05,3
AD_Bellenguez_noproxy_2026,chr2_125689597_127728648,susie_rss_2,2.867,4.15e-03,2
AD_Wightman_Excluding23andMe_2021,chr11_119829118_122612025,susie_rss_2,4.140,3.47e-05,2
AD_Wightman_Excluding23andMe_2021,chr20_44760233_46209985,susie_rss_1,4.411,1.03e-05,1
AD_Wightman_Excluding23andMe_2021,chr8_26225312_29469590,susie_rss_3,4.253,2.11e-05,3


In [6]:
## ---- inputs for the [cojo] step ------------------------------------------------
cojo_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_COJO"
dir.create(file.path(cojo_dir, "input"), recursive = TRUE, showWarnings = FALSE)

## every CS in study x region pairs with multiple SuSiE-rRSS credible sets (the [cojo] step runs one task per pair)
multi_cs <- cs_tab %>%
  filter(n_cs_total > 1) %>%
  arrange(study, block_id, cs) %>%
  transmute(study, block_id, cs,
            max_abs_z = round(max_abs_z, 3),        # strongest marginal |z| in the CS
            p_of_maxz = signif(p_of_maxz, 3),
            n_cs_total)
write_tsv(multi_cs, file.path(cojo_dir, "input/multi_cs.tsv"))

## every eb_mix CS variant in those study x region pairs, for the joint model (top PIP per CS) and the overlap
eb_mix %>%
  semi_join(distinct(multi_cs, study, block_id), by = c("study", "block_id")) %>%
  filter(!is.na(cs_coverage_0.95), !cs_coverage_0.95 %in% c("0", "susie_rss_0")) %>%
  transmute(study, block_id, cs = cs_coverage_0.95, variant_ID, z, PIP) %>%
  write_tsv(file.path(cojo_dir, "input/multi_cs_region_variants.tsv"))

nrow(multi_cs); n_distinct(paste(multi_cs$study, multi_cs$block_id))    # study x region pairs: 56 expected

## the earlier weak-CS list (max |z| below p = 1e-5) is a subset, kept for reference
weak_cs <- weak_list(1e-5)
write_tsv(weak_cs, file.path(cojo_dir, "input/weak_cs_p1e-5.tsv"))

[1] 121

[1] 56

## COJO job

`[cojo]`: one task per study × region with multiple SuSiE-rRSS credible sets (`input/multi_cs.tsv`). It writes `<unit>.ma` and the LD, then

- **joint** (main): `cojo joint` on the top-PIP variant of every eb_mix CS in the region, fitted together, with no p-value filter and no selection → `<unit>.cojo_joint.tsv` (one row per CS: marginal z, joint beta/se/z/p). All CS variants together would be collinear (within-CS LD is high), so each CS enters through one variant.
- **slct** (secondary): `cojo slct` once per `--p-cutoffs` value (default 5e-8 and 1e-6) → `<unit>.cojo_summary.tsv`, the selected SNPs against every eb_mix CS (`snp_in_cs`, `max_r2_cs`, and `stop_min_p`, the smallest p-value left when selection stopped).

Output: `/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_COJO/cojo_multi_cs/result/<study>.<block_id>/`

In [ ]:
[cojo]
# cojopy on each study x region pair with a weak SuSiE-rRSS CS.
# Inputs as in the manual checks below: z from the eb_mix sumstats, b/se imputed with freq = 0.5,
# LD from the ADSP R4 EUR sketch.
#   joint: one model on the top-PIP variant of every eb_mix CS in the region (no p-value filter, no selection)
#   slct : stepwise selection, one run per p-value cutoff
parameter: weak_cs = path
# weak_cs_p1e-5.tsv (study, block_id, cs, ...)
parameter: cs_variants = path
# weak_region_cs_variants.tsv (study, block_id, cs, variant_ID, z, PIP)
parameter: sumstat_dir = path("/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10/AD_GWAS_rss_ebmix_r4_L10/output/sumstats")
parameter: ld_meta_file = path("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/rss_ld_sketch/ld_meta_file.tsv")
parameter: out_dir = path("output")
parameter: p_cutoffs = [5e-8, 1e-6]
# slct only: cojopy default (5e-8) and a lower one
parameter: collinear_cutoff = 0.9
# cojopy default
parameter: cojo = "cojo"
# cojopy executable
parameter: keep_ld = False
# keep the full <unit>.ld text matrix (~5k x 5k) after the runs
parameter: job_size = 1
parameter: walltime = "4h"
parameter: mem = "16G"
parameter: numThreads = 1
parameter: container = ""

import csv
with open(weak_cs) as f:
    units = list(dict.fromkeys(f"{r['study']}.{r['block_id']}" for r in csv.DictReader(f, delimiter="\t")))

input: for_each = "units"
output: f'{out_dir:a}/{_units}/{_units}.cojo_joint.tsv', f'{out_dir:a}/{_units}/{_units}.cojo_summary.tsv'

task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_units}'

R: expand = '${ }', stdout = f"{_output[0]:n}.prep.stdout", stderr = f"{_output[0]:n}.prep.stderr", container = container
    library(tidyverse)
    library(pecotmr)
    devtools::load_all("/hpc/users/sunh17/haochen/rss_qc/pecotmr_latest/pecotmr", compile = FALSE)

    unit     <- "${_units}"
    out_d    <- ${_output[0]:dr}
    study    <- str_split_fixed(unit, "\\.", 2)[, 1]
    block_id <- str_split_fixed(unit, "\\.", 2)[, 2]
    blk      <- str_split_fixed(block_id, "_", 3)
    region   <- sprintf("%s:%s-%s", blk[, 1], blk[, 2], blk[, 3])
    cat("unit:", unit, " region:", region, "\n")

    dat = readRDS(file.path(${sumstat_dir:r}, paste0(unit, ".gwas_sumstats.rds")))
    sumstat = getSumStatsDf(dat)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)

    select <- dplyr::select
    ld_meta_file <- ${ld_meta_file:r}
        res_sketch           <- load_LD_matrix(ld_meta_file, region, return_genotype = TRUE)
        res_sketch$LD_matrix <- scale(res_sketch$LD_matrix, center = TRUE, scale = TRUE)
        sketch_ref           <- res_sketch$ref_panel

    ske = res_sketch$LD_matrix

    LD = crossprod(ske) / (nrow(ske) - 1)

    z  <- sumstat$z
    N  <- sumstat$N
    p0 <- 0.5                                   # constant freq -> z/R-only conditional analysis
    se <- 1 / sqrt(2 * p0 * (1 - p0) * (N + z^2))   # = 1/sqrt(0.5*(N + z^2))
    b  <- z * se
    pv <- 2 * pnorm(-abs(z))

    ma <- tibble(SNP  = sumstat$variant_id,
                 A1   = sumstat$A1,            # effect allele (z is w.r.t. A1)
                 A2   = sumstat$A2,
                 freq = p0,
                 b    = b,
                 se   = se,
                 p    = pv,
                 N    = N)
    write_tsv(ma, file.path(out_d, paste0(unit, ".ma")))

    R = LD[ma$SNP, ma$SNP]
    stopifnot(nrow(R) == nrow(ma))              # same order & length as sumstat
    write.table(R, file.path(out_d, paste0(unit, ".ld")), sep = "\t",
                row.names = FALSE, col.names = FALSE, quote = FALSE)

    ## LD between every SNP and the eb_mix CS variants of this region (for the overlap check)
    cs_var <- read_tsv(${cs_variants:r}, show_col_types = FALSE) %>%
      filter(study == !!study, block_id == !!block_id)
    miss <- setdiff(cs_var$variant_ID, ma$SNP)
    cat(sprintf("%d SNPs, max |z| = %.3f; %d CS variants (%d not in LD)\n",
                nrow(ma), max(abs(z)), nrow(cs_var), length(miss)))
    cs_var <- filter(cs_var, variant_ID %in% ma$SNP)
    saveRDS(list(cs_var = cs_var, R = R[, cs_var$variant_ID, drop = FALSE]),
            file.path(out_d, paste0(unit, ".cs_ld.rds")))

    ## one variant per CS for the joint model: the top PIP (SuSiE-rRSS's own pick)
    cs_lead <- cs_var %>% group_by(cs) %>% slice_max(PIP, n = 1, with_ties = FALSE) %>% ungroup()
    writeLines(cs_lead$variant_ID, file.path(out_d, paste0(unit, ".cs_leads.txt")))

bash: expand = '${ }', stdout = f"{_output[0]:n}.cojo.stdout", stderr = f"{_output[0]:n}.cojo.stderr", container = container
    cd ${_output[0]:dr}
    export COLUMNS=250                          # keep the rich log on one line per message
    ${cojo} joint --sumstats ${_units}.ma --ld-matrix ${_units}.ld \
              --extract-snps ${_units}.cs_leads.txt --output ${_units}.joint.txt \
              > ${_units}.joint.log 2>&1
    for p in ${" ".join(str(x) for x in p_cutoffs)}; do
      ${cojo} slct --sumstats ${_units}.ma --ld-matrix ${_units}.ld \
                --output ${_units}.p$p.slct.txt --p-cutoff $p --collinear-cutoff ${collinear_cutoff} \
                > ${_units}.p$p.log 2>&1
    done
    ${'' if keep_ld else 'rm -f ' + _units + '.ld'}

R: expand = '${ }', stdout = f"{_output[0]:n}.summary.stdout", stderr = f"{_output[0]:n}.summary.stderr", container = container
    library(tidyverse)
    unit   <- "${_units}"
    out_d  <- ${_output[0]:dr}
    p_tags <- c(${", ".join(repr(str(x)) for x in p_cutoffs)})
    cs_ld  <- readRDS(file.path(out_d, paste0(unit, ".cs_ld.rds")))
    cs_var <- cs_ld$cs_var

    ## one row per (p cutoff, COJO-selected SNP, eb_mix CS in this region)
    summ <- map_dfr(p_tags, function(pt) {
      f    <- file.path(out_d, sprintf("%s.p%s.slct.txt", unit, pt))
      logl <- readLines(file.path(out_d, sprintf("%s.p%s.log", unit, pt)))
      ## p-value of the best SNP left when selection stopped
      stop_p <- str_match(logl, "minimum (conditional )?p-value: ([0-9.eE+-]+)")[, 3]
      stop_p <- dplyr::last(as.numeric(na.omit(stop_p)))
      sel <- tryCatch(read_tsv(f, show_col_types = FALSE), error = function(e) tibble())
      if (!"SNP" %in% names(sel)) sel <- tibble(SNP = character(), original_p = numeric(), joint_p = numeric())
      base <- tibble(p_cutoff = as.numeric(pt), n_selected = nrow(sel), stop_min_p = stop_p)
      if (nrow(sel) == 0) return(base)
      crossing(sel %>% select(SNP, original_p, joint_beta, joint_se, joint_p),
               cs = unique(cs_var$cs)) %>%
        rowwise() %>%
        mutate(snp_in_cs = SNP %in% cs_var$variant_ID[cs_var$cs == cs],
               max_r2_cs = if (SNP %in% rownames(cs_ld$R))
                             max(cs_ld$R[SNP, cs_var$variant_ID[cs_var$cs == cs]]^2) else NA_real_) %>%
        ungroup() %>%
        bind_cols(base[rep(1, nrow(.)), ], .)
    }) %>%
      mutate(study    = str_split_fixed(unit, "\\.", 2)[, 1],
             block_id = str_split_fixed(unit, "\\.", 2)[, 2], .before = 1)

    print(as.data.frame(summ))
    write_tsv(summ, ${_output[1]:r})

    ## joint model: one row per CS (its top-PIP variant), fitted together with the other CSs' leads
    cs_lead <- cs_var %>% group_by(cs) %>% slice_max(PIP, n = 1, with_ties = FALSE) %>% ungroup()
    joint <- read_tsv(file.path(out_d, paste0(unit, ".joint.txt")), show_col_types = FALSE) %>%
      mutate(joint_z = joint_beta / joint_se)
    joint <- cs_lead %>%
      transmute(study, block_id, cs, n_cs = nrow(cs_lead), SNP = variant_ID, PIP, marginal_z = z) %>%
      left_join(select(joint, SNP, original_p, joint_beta, joint_se, joint_z, joint_p), by = "SNP")

    print(as.data.frame(joint))
    write_tsv(joint, ${_output[0]:r})

## Submit

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_COJO"
job_name="cojo_multi_cs"
job_dir="${base_dir}/${job_name}"

# Inputs written by the R cell above
input_dir="${base_dir}/input"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base LSF script template =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=16000]
#BSUB -W 48:00
#BSUB -o WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/24.Fig6FG.GCTA_COJO_Weak_CS.ipynb" cojo \
    --weak-cs INPUT_DIR/multi_cs.tsv \
    --cs-variants INPUT_DIR/multi_cs_region_variants.tsv \
    --out-dir WORK_DIR/result/ \
    --p-cutoffs 5e-8 1e-6 \
    --mem 16G -J 4
EOF
chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

output_script="${job_name}.sh.submit"
sed -e "s|WORK_DIR|${work_dir}|g" \
    -e "s|INPUT_DIR|${input_dir}|g" \
    -e "s|JOB_NAME|${job_name}|g" \
    "${base_sh}" > "${output_script}"
 sb "${output_script}"

## SuSiE-RSS multi-CS regions

Use the same `[cojo]` joint model to assess additional credible sets in studies and regions where SuSiE-RSS (`AD_GWAS_rss_baseline_r4_L10`) reports multiple sets. Every credible set in those pairs enters through its top-PIP variant, as for SuSiE-rRSS.

The SuSiE-RSS run reads its inputs from `ADGWAS_COJO/input_susie_rss/` and writes results to `ADGWAS_COJO/cojo_multi_cs_susie_rss/result/<study>.<block_id>/`. The summary cell below reads both runs and writes `ADGWAS_COJO/cojo_joint_rrss_vs_susie_rss.tsv`.

In [7]:
## ---- inputs for [cojo] on the SuSiE-RSS (baseline) multi-CS pairs ----------------
## needs read_gwas_tables(), cojo_dir and multi_cs from the cells above
rss_root  <- "/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10/AD_GWAS_rss_baseline_r4_L10"
rss_input <- file.path(cojo_dir, "input_susie_rss")
dir.create(rss_input, recursive = TRUE, showWarnings = FALSE)

susie_rss <- read_gwas_tables(file.path(rss_root, "export/")) %>%
  filter(!is.na(cs_coverage_0.95), !cs_coverage_0.95 %in% c("", "0", "susie_rss_0"))

## every CS in study x region pairs with multiple SuSiE-RSS credible sets
rss_multi_cs <- susie_rss %>%
  group_by(study, block_id, cs = cs_coverage_0.95) %>%
  summarise(max_abs_z = max(abs(z), na.rm = TRUE), .groups = "drop") %>%
  group_by(study, block_id) %>% mutate(n_cs_total = n_distinct(cs)) %>% ungroup() %>%
  filter(n_cs_total > 1) %>%
  arrange(study, block_id, cs) %>%
  transmute(study, block_id, cs,
            max_abs_z = round(max_abs_z, 3),
            p_of_maxz = signif(2 * pnorm(-max_abs_z), 3),
            n_cs_total)
write_tsv(rss_multi_cs, file.path(rss_input, "multi_cs.tsv"))

susie_rss %>%
  semi_join(distinct(rss_multi_cs, study, block_id), by = c("study", "block_id")) %>%
  transmute(study, block_id, cs = cs_coverage_0.95, variant_ID, z, PIP) %>%
  write_tsv(file.path(rss_input, "multi_cs_region_variants.tsv"))

## CSs, study x region pairs, and how many of the pairs are also multi-CS under SuSiE-rRSS
rss_pairs <- distinct(rss_multi_cs, study, block_id)
tibble(n_cs = nrow(rss_multi_cs), n_pairs = nrow(rss_pairs),
       n_pairs_also_rrss_multi = nrow(semi_join(rss_pairs, multi_cs, by = c("study", "block_id"))))
count(rss_multi_cs %>% distinct(study, block_id, n_cs_total), n_cs_total)

n_cs,n_pairs,n_pairs_also_rrss_multi
<int>,<int>,<int>
235,82,56


n_cs_total,n
<int>,<int>
2,58
3,11
4,3
5,1
7,6
9,3


In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_COJO"
job_name="cojo_multi_cs_susie_rss"
job_dir="${base_dir}/${job_name}"

# Inputs written by the SuSiE-RSS R cell above
input_dir="${base_dir}/input_susie_rss"

# z from the SuSiE-RSS run's own sumstats when present, else the eb_mix ones (the [cojo] default)
finemap_dir="/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10"
sumstat_dir="${finemap_dir}/AD_GWAS_rss_baseline_r4_L10/output/sumstats"
[ -d "${sumstat_dir}" ] || sumstat_dir="${finemap_dir}/AD_GWAS_rss_ebmix_r4_L10/output/sumstats"
echo "sumstats: ${sumstat_dir}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base LSF script template =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=16000]
#BSUB -W 48:00
#BSUB -o WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/24.Fig6FG.GCTA_COJO_Weak_CS.ipynb" cojo \
    --weak-cs INPUT_DIR/multi_cs.tsv \
    --cs-variants INPUT_DIR/multi_cs_region_variants.tsv \
    --sumstat-dir SUMSTAT_DIR \
    --out-dir WORK_DIR/result/ \
    --p-cutoffs 5e-8 1e-6 \
    --mem 16G -J 4
EOF
chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

output_script="${job_name}.sh.submit"
sed -e "s|WORK_DIR|${work_dir}|g" \
    -e "s|INPUT_DIR|${input_dir}|g" \
    -e "s|SUMSTAT_DIR|${sumstat_dir}|g" \
    -e "s|JOB_NAME|${job_name}|g" \
    "${base_sh}" > "${output_script}"
 sb "${output_script}"

### Joint support of the additional credible sets: SuSiE-rRSS vs SuSiE-RSS

Run after both jobs finish. In each study × region pair, the **lead** set is the one whose top-PIP variant has the largest marginal |z|; the remaining sets are **additional**. For each method, count additional sets with joint P < 5 × 10⁻⁸ and < 10⁻⁵ and sets with stronger joint than marginal associations. Also report the joint-to-marginal effect ratio |β_joint| / |β_marginal| (marginal β from `<unit>.ma`). SuSiE-RSS sets are split by `in_rrss`: whether the set shares a variant with a SuSiE-rRSS set in the same study × region (sets only SuSiE-RSS reports have `in_rrss = FALSE`).

In [8]:
library(tidyverse)
## ---- joint support of additional CSs, both runs (needs eb_mix, susie_rss, rss_multi_cs from above) ----
mlog10p <- function(z) -(log(2) + pnorm(-abs(z), log.p = TRUE)) / log(10)
read_joint <- function(res_dir, method)
  list.files(res_dir, pattern = "\\.cojo_joint\\.tsv$", recursive = TRUE, full.names = TRUE) %>%
    map_dfr(function(f) {
      ma <- read_tsv(sub("\\.cojo_joint\\.tsv$", ".ma", f), show_col_types = FALSE) %>% select(SNP, marginal_beta = b)
      read_tsv(f, show_col_types = FALSE) %>% left_join(ma, by = "SNP")
    }) %>%
    mutate(method = method)

## SuSiE-RSS sets that share a variant with a SuSiE-rRSS set in the same study x region
rrss_cs_var <- eb_mix %>%
  filter(!is.na(cs_coverage_0.95), !cs_coverage_0.95 %in% c("", "0", "susie_rss_0")) %>%
  distinct(study, block_id, variant_ID) %>% mutate(hit = TRUE)
rss_in_rrss <- susie_rss %>%
  semi_join(distinct(rss_multi_cs, study, block_id), by = c("study", "block_id")) %>%
  transmute(study, block_id, cs = cs_coverage_0.95, variant_ID) %>%
  left_join(rrss_cs_var, by = c("study", "block_id", "variant_ID")) %>%
  group_by(study, block_id, cs) %>% summarise(in_rrss = any(hit %in% TRUE), .groups = "drop") %>%
  mutate(method = "SuSiE-RSS")

joint_all <- bind_rows(read_joint(file.path(cojo_dir, "cojo_multi_cs/result"), "SuSiE-rRSS"),
                       read_joint(file.path(cojo_dir, "cojo_multi_cs_susie_rss/result"), "SuSiE-RSS")) %>%
  group_by(method, study, block_id) %>%
  mutate(role = if_else(row_number(desc(abs(marginal_z))) == 1, "lead", "additional")) %>%
  ungroup() %>%
  left_join(rss_in_rrss, by = c("method", "study", "block_id", "cs")) %>%
  mutate(in_rrss          = if_else(method == "SuSiE-rRSS", TRUE, in_rrss),
         marginal_mlog10p = mlog10p(marginal_z),
         joint_mlog10p    = mlog10p(joint_z),
         beta_ratio       = abs(joint_beta) / abs(marginal_beta))

## pairs per method: did every pair finish?
joint_all %>% group_by(method) %>%
  summarise(n_pairs = n_distinct(paste(study, block_id)), n_cs = n(), n_joint_missing = sum(is.na(joint_z)))

## additional sets: joint support
joint_all %>% filter(role == "additional") %>%
  group_by(method, in_rrss) %>%
  summarise(n_cs              = n(),
            joint_p_gws       = sum(joint_mlog10p > -log10(5e-8), na.rm = TRUE),
            joint_p_1e5       = sum(joint_mlog10p > -log10(1e-5), na.rm = TRUE),
            marginal_p_gws    = sum(marginal_mlog10p > -log10(5e-8), na.rm = TRUE),
            stronger_jointly  = sum(joint_mlog10p > marginal_mlog10p, na.rm = TRUE),
            median_beta_ratio = round(median(beta_ratio, na.rm = TRUE), 2),
            .groups = "drop")

write_tsv(joint_all, file.path(cojo_dir, "cojo_joint_rrss_vs_susie_rss.tsv"))

method,n_pairs,n_cs,n_joint_missing
<chr>,<int>,<int>,<int>
SuSiE-RSS,82,235,0
SuSiE-rRSS,56,121,0


method,in_rrss,n_cs,joint_p_gws,joint_p_1e5,marginal_p_gws,stronger_jointly,median_beta_ratio
<chr>,<lgl>,<int>,<int>,<int>,<int>,<int>,<dbl>
SuSiE-RSS,FALSE,89,58,65,70,22,0.81
SuSiE-RSS,TRUE,64,54,64,39,51,1.06
SuSiE-rRSS,TRUE,65,55,65,40,52,1.05


In [9]:
apoe_block <- "chr19_42346101_46842901"
joint_all %>% filter(role == "additional") %>%
  mutate(locus = if_else(block_id == apoe_block, "APOE", "non-APOE")) %>%
  group_by(locus, method, in_rrss) %>%
  summarise(n_pairs = n_distinct(paste(study, block_id)), n_cs = n(),
            joint_p_gws = sum(joint_mlog10p > -log10(5e-8), na.rm = TRUE),
            joint_p_1e5 = sum(joint_mlog10p > -log10(1e-5), na.rm = TRUE),
            marginal_p_gws = sum(marginal_mlog10p > -log10(5e-8), na.rm = TRUE),
            stronger_jointly = sum(joint_mlog10p > marginal_mlog10p, na.rm = TRUE),
            median_beta_ratio = round(median(beta_ratio, na.rm = TRUE), 2), .groups = "drop")

locus,method,in_rrss,n_pairs,n_cs,joint_p_gws,joint_p_1e5,marginal_p_gws,stronger_jointly,median_beta_ratio
<chr>,<chr>,<lgl>,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>
APOE,SuSiE-RSS,FALSE,10,54,49,49,50,13,0.77
non-APOE,SuSiE-RSS,FALSE,21,35,9,16,20,9,0.87
non-APOE,SuSiE-RSS,TRUE,56,64,54,64,39,51,1.06
non-APOE,SuSiE-rRSS,TRUE,56,65,55,65,40,52,1.05


## Stepwise selection only (previous version)

`[cojo_slct]`: the earlier `[cojo]` step, `cojo slct` only, once per `--p-cutoffs` value → `<unit>.cojo_summary.tsv`. Same inputs as `[cojo]`; submitted as its own job (`cojo_weak_cs_slct`), so outputs go to `/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_COJO/cojo_weak_cs_slct/result/<study>.<block_id>/`.

In [ ]:
[cojo_slct]
# previous version: cojopy slct only (stepwise selection), one run per p-value cutoff
parameter: weak_cs = path
# weak_cs_p1e-5.tsv (study, block_id, cs, ...)
parameter: cs_variants = path
# weak_region_cs_variants.tsv (study, block_id, cs, variant_ID, z)
parameter: sumstat_dir = path("/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10/AD_GWAS_rss_ebmix_r4_L10/output/sumstats")
parameter: ld_meta_file = path("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/rss_ld_sketch/ld_meta_file.tsv")
parameter: out_dir = path("output")
parameter: p_cutoffs = [5e-8, 1e-6]
# cojopy default (5e-8) and a lower one
parameter: collinear_cutoff = 0.9
# cojopy default
parameter: cojo = "cojo"
# cojopy executable
parameter: keep_ld = False
# keep the full <unit>.ld text matrix (~5k x 5k) after the runs
parameter: job_size = 1
parameter: walltime = "4h"
parameter: mem = "16G"
parameter: numThreads = 1
parameter: container = ""

import csv
with open(weak_cs) as f:
    units = list(dict.fromkeys(f"{r['study']}.{r['block_id']}" for r in csv.DictReader(f, delimiter="\t")))

input: for_each = "units"
output: f'{out_dir:a}/{_units}/{_units}.cojo_summary.tsv'

task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_units}'

R: expand = '${ }', stdout = f"{_output:n}.prep.stdout", stderr = f"{_output:n}.prep.stderr", container = container
    library(tidyverse)
    library(pecotmr)
    devtools::load_all("/hpc/users/sunh17/haochen/rss_qc/pecotmr_latest/pecotmr", compile = FALSE)

    unit     <- "${_units}"
    out_d    <- ${_output:dr}
    study    <- str_split_fixed(unit, "\\.", 2)[, 1]
    block_id <- str_split_fixed(unit, "\\.", 2)[, 2]
    blk      <- str_split_fixed(block_id, "_", 3)
    region   <- sprintf("%s:%s-%s", blk[, 1], blk[, 2], blk[, 3])
    cat("unit:", unit, " region:", region, "\n")

    dat = readRDS(file.path(${sumstat_dir:r}, paste0(unit, ".gwas_sumstats.rds")))
    sumstat = getSumStatsDf(dat)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)

    select <- dplyr::select
    ld_meta_file <- ${ld_meta_file:r}
        res_sketch           <- load_LD_matrix(ld_meta_file, region, return_genotype = TRUE)
        res_sketch$LD_matrix <- scale(res_sketch$LD_matrix, center = TRUE, scale = TRUE)
        sketch_ref           <- res_sketch$ref_panel

    ske = res_sketch$LD_matrix

    LD = crossprod(ske) / (nrow(ske) - 1)

    z  <- sumstat$z
    N  <- sumstat$N
    p0 <- 0.5                                   # constant freq -> z/R-only conditional analysis
    se <- 1 / sqrt(2 * p0 * (1 - p0) * (N + z^2))   # = 1/sqrt(0.5*(N + z^2))
    b  <- z * se
    pv <- 2 * pnorm(-abs(z))

    ma <- tibble(SNP  = sumstat$variant_id,
                 A1   = sumstat$A1,            # effect allele (z is w.r.t. A1)
                 A2   = sumstat$A2,
                 freq = p0,
                 b    = b,
                 se   = se,
                 p    = pv,
                 N    = N)
    write_tsv(ma, file.path(out_d, paste0(unit, ".ma")))

    R = LD[ma$SNP, ma$SNP]
    stopifnot(nrow(R) == nrow(ma))              # same order & length as sumstat
    write.table(R, file.path(out_d, paste0(unit, ".ld")), sep = "\t",
                row.names = FALSE, col.names = FALSE, quote = FALSE)

    ## LD between every SNP and the eb_mix CS variants of this region (for the overlap check)
    cs_var <- read_tsv(${cs_variants:r}, show_col_types = FALSE) %>%
      filter(study == !!study, block_id == !!block_id)
    miss <- setdiff(cs_var$variant_ID, ma$SNP)
    cat(sprintf("%d SNPs, max |z| = %.3f; %d CS variants (%d not in LD)\n",
                nrow(ma), max(abs(z)), nrow(cs_var), length(miss)))
    cs_var <- filter(cs_var, variant_ID %in% ma$SNP)
    saveRDS(list(cs_var = cs_var, R = R[, cs_var$variant_ID, drop = FALSE]),
            file.path(out_d, paste0(unit, ".cs_ld.rds")))

bash: expand = '${ }', stdout = f"{_output:n}.cojo.stdout", stderr = f"{_output:n}.cojo.stderr", container = container
    cd ${_output:dr}
    export COLUMNS=250                          # keep the rich log on one line per message
    for p in ${" ".join(str(x) for x in p_cutoffs)}; do
      ${cojo} slct --sumstats ${_units}.ma --ld-matrix ${_units}.ld \
                --output ${_units}.p$p.slct.txt --p-cutoff $p --collinear-cutoff ${collinear_cutoff} \
                > ${_units}.p$p.log 2>&1
    done
    ${'' if keep_ld else 'rm -f ' + _units + '.ld'}

R: expand = '${ }', stdout = f"{_output:n}.summary.stdout", stderr = f"{_output:n}.summary.stderr", container = container
    library(tidyverse)
    unit   <- "${_units}"
    out_d  <- ${_output:dr}
    p_tags <- c(${", ".join(repr(str(x)) for x in p_cutoffs)})
    cs_ld  <- readRDS(file.path(out_d, paste0(unit, ".cs_ld.rds")))
    cs_var <- cs_ld$cs_var

    ## one row per (p cutoff, COJO-selected SNP, eb_mix CS in this region)
    summ <- map_dfr(p_tags, function(pt) {
      f    <- file.path(out_d, sprintf("%s.p%s.slct.txt", unit, pt))
      logl <- readLines(file.path(out_d, sprintf("%s.p%s.log", unit, pt)))
      ## p-value of the best SNP left when selection stopped
      stop_p <- str_match(logl, "minimum (conditional )?p-value: ([0-9.eE+-]+)")[, 3]
      stop_p <- dplyr::last(as.numeric(na.omit(stop_p)))
      sel <- tryCatch(read_tsv(f, show_col_types = FALSE), error = function(e) tibble())
      if (!"SNP" %in% names(sel)) sel <- tibble(SNP = character(), original_p = numeric(), joint_p = numeric())
      base <- tibble(p_cutoff = as.numeric(pt), n_selected = nrow(sel), stop_min_p = stop_p)
      if (nrow(sel) == 0) return(base)
      crossing(sel %>% select(SNP, original_p, joint_beta, joint_se, joint_p),
               cs = unique(cs_var$cs)) %>%
        rowwise() %>%
        mutate(snp_in_cs = SNP %in% cs_var$variant_ID[cs_var$cs == cs],
               max_r2_cs = if (SNP %in% rownames(cs_ld$R))
                             max(cs_ld$R[SNP, cs_var$variant_ID[cs_var$cs == cs]]^2) else NA_real_) %>%
        ungroup() %>%
        bind_cols(base[rep(1, nrow(.)), ], .)
    }) %>%
      mutate(study    = str_split_fixed(unit, "\\.", 2)[, 1],
             block_id = str_split_fixed(unit, "\\.", 2)[, 2], .before = 1)

    print(as.data.frame(summ))
    write_tsv(summ, ${_output:r})

Submit

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_COJO"
job_name="cojo_multi_cs_slct"
job_dir="${base_dir}/${job_name}"

# Inputs written by the R cell above
input_dir="${base_dir}/input"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base LSF script template =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=16000]
#BSUB -W 48:00
#BSUB -o WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/24.Fig6FG.GCTA_COJO_Weak_CS.ipynb" cojo_slct \
    --weak-cs INPUT_DIR/multi_cs.tsv \
    --cs-variants INPUT_DIR/multi_cs_region_variants.tsv \
    --out-dir WORK_DIR/result/ \
    --p-cutoffs 5e-8 1e-6 \
    --mem 16G -J 4
EOF
chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

output_script="${job_name}.sh.submit"
sed -e "s|WORK_DIR|${work_dir}|g" \
    -e "s|INPUT_DIR|${input_dir}|g" \
    -e "s|JOB_NAME|${job_name}|g" \
    "${base_sh}" > "${output_script}"
 sb "${output_script}"